# Modelo A — Vecinos por perfil

**Hipótesis:** las personas con características parecidas tienden a valorar positivamente productos similares.

**Método:** One-Hot de los cuatro atributos → distancia coseno → K usuarios vecinos → votos positivos por producto → Top-10.

**Ejemplo conceptual:** para piel seca, tono claro, ojos marrones y cabello castaño, buscamos las personas más parecidas. Si 18 vecinos valoraron positivamente la crema X y 12 la crema Y, X queda antes que Y. Las cifras del ejemplo son ilustrativas.

Este es el enfoque de tu antiguo Modelo 2. K es cantidad de personas, no cantidad de productos. No se usan ratings negativos para el ranking y cada vecino tiene el mismo peso.

**Estado:** listo para ejecutar en tu proyecto. Las métricas finales están pendientes; no se incluyen resultados simulados como resultados reales.

## 1. Librerías, datos y split común

Copiá `sephora_common.py` junto a estos notebooks en `notebooks/`. Ese archivo contiene las reglas compartidas de preparación, partición y evaluación. El algoritmo particular de este modelo aparece completo abajo.

Se leen `data/processed/reviews_clean.parquet` y `products_clean.parquet`, con alternativa CSV si no existe el parquet. Ejecutar desde arriba con un kernel que tenga las dependencias instaladas.

In [1]:
from pathlib import Path
import sys
import heapq
from itertools import islice
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# El archivo sephora_common.py debe estar junto a los notebooks.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data" / "processed").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abrí este notebook dentro del proyecto que contiene data/processed")
helper_dir = next((p for p in [Path.cwd(), PROJECT_ROOT / "notebooks"]
                   if (p / "sephora_common.py").exists()), None)
if helper_dir is None:
    raise FileNotFoundError("Copiá sephora_common.py a notebooks/ junto a estos archivos")
sys.path.insert(0, str(helper_dir))
import sephora_common as sc
from sephora_common import PROFILE_COLS, rank_scores, text_value

# Los mismos valores en los cuatro notebooks.
N_VALIDACION = 500
N_TEST = 500
SEMILLA = 42
ctx = sc.load_context(N_VALIDACION, N_TEST, SEMILLA)
print("Protocolo:", ctx.protocol_id[:12])
print("Resultados:", ctx.output_dir)
display(pd.Series({k:v for k,v in ctx.manifest.items()
                   if k not in ["validation_cases", "test_cases", "versions"]}).to_frame("valor"))
display(pd.DataFrame(ctx.manifest["versions"], index=[0]))

Protocolo: 99fc64b0ea63
Resultados: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63


,valor
version,cold-start-users-v2.0
seed,42
protocol_id,99fc64b0ea63f0e1e20f6a8a2d96aee1222e401c635a55...
reviews_input,1093855
reviews_valid_before_pair_dedup,1093855
reviews_unique_pairs,1088886
removed_duplicate_pairs,4969
catalog_skincare,2420
users_total,503216
eligible_users,161396


,python,numpy,pandas,scipy,scikit_learn
0,3.14.5,2.4.6,3.0.3,1.18.1,1.9.0


## 2. Condiciones comunes de comparación

- **Consulta:** tipo y tono de piel, ojos y cabello. Cada enfoque decide qué atributos utiliza. Ninguno recibe compras ni ratings del usuario de consulta.
- **Candidatos:** catálogo Skincare, idéntico para todos. Un modelo basado en evidencia puede puntuar solo parte de este catálogo; ese límite se refleja en la cobertura.
- **Positivo:** `rating >= 4` **o** `is_recommended = True`. La misma regla sirve para todos. Una review de rating bajo pero recomendada es positiva según esta definición.
- **Datos:** un par usuario–producto por vez; se conserva la última review con fecha disponible. Se cuantifica lo descartado. Los perfiles se resumen por moda, sin utilizar ratings para definirlos.
- **Validación:** 500 usuarios para elegir hiperparámetros.
- **Test:** otros 500 usuarios para evaluar la configuración elegida. Se fija un producto positivo por usuario.
- **Sin fuga:** ninguna review de esos 1.000 usuarios entra en entrenamiento, estadísticas de ratings, popularidad o votos.
- **Cold start:** los demás positivos del usuario también permanecen ocultos. No se utilizan ni siquiera para filtrar el Top-10.

Se seleccionan usuarios con al menos dos productos positivos y tipo/tono conocidos. Es una evaluación en esa población, no en todos los visitantes del sitio. Se supone que los atributos personales aportados en las reviews son conocidos al consultar. Esta partición no simula el paso del tiempo.

La huella del protocolo verifica que datos, semilla y tamaños sean iguales. Si cambian, los resultados van a otra carpeta; el comparador no mezcla protocolos.

## 3. Cómo leer el algoritmo

1. `__init__` prepara perfiles de usuarios de entrenamiento e indexa sus productos positivos.
2. `neighbors` calcula similitud coseno. Para ahorrar tiempo compara perfiles únicos y luego recupera las personas correspondientes. Es la misma búsqueda de vecinos, no clustering.
3. Los empates de distancia se resuelven por ID de usuario para que la selección sea reproducible.
4. `recommend` suma votos de esos vecinos. Los empates de votos se resuelven por ID de producto.

**Limitación:** muchas personas comparten exactamente el mismo perfil; elegir solo K puede usar un subconjunto limitado de la evidencia. Tampoco garantiza compatibilidad dermatológica.

In [2]:
class ProfileKNN:
    """K vecinos exactos por coseno; agrupa perfiles repetidos para acelerar."""
    def __init__(self, ctx):
        self.profiles = ctx.profiles.loc[sorted(ctx.train.author_id.unique())]
        self.encoder = DictVectorizer(sparse=True)
        records = self.profiles.to_dict("records")
        self.encoder.fit(records + [{c: "Unknown" for c in PROFILE_COLS}])
        self.allowed = {c: set(self.profiles[c]) | {"Unknown"} for c in PROFILE_COLS}
        # Una firma representa a todas las personas con exactamente ese perfil.
        self.groups = {}
        for uid, row in self.profiles.iterrows():
            self.groups.setdefault(tuple(row[c] for c in PROFILE_COLS), []).append(uid)
        self.signatures = list(self.groups)
        self.X = self.encoder.transform([dict(zip(PROFILE_COLS, s)) for s in self.signatures])
        self.positives = ctx.train[ctx.train.positive].groupby("author_id").product_id.apply(list).to_dict()
        self.cache = {}
        self.n_train_users = len(self.profiles)

    def neighbors(self, profile, k):
        if not 1 <= k <= self.n_train_users:
            raise ValueError(f"K={k} inválido para {self.n_train_users} usuarios de entrenamiento")
        key = tuple(profile.get(c, "Unknown") if profile.get(c, "Unknown") in self.allowed[c] else "Unknown" for c in PROFILE_COLS)
        if key not in self.cache or len(self.cache[key]) < k:
            query = self.encoder.transform([dict(zip(PROFILE_COLS, key))])
            distances = np.round(1-cosine_similarity(query, self.X).ravel(), 12)
            neighbors = []
            for d in sorted(set(distances)):
                group_indices = np.flatnonzero(distances == d)
                # Cada lista ya está ordenada por author_id. El merge resuelve empates.
                streams = [self.groups[self.signatures[i]] for i in group_indices]
                remaining = k-len(neighbors)
                neighbors.extend(islice(heapq.merge(*streams), remaining))
                if len(neighbors) == k:
                    break
            self.cache[key] = neighbors
        return self.cache[key][:k]

    def recommend(self, profile, config, top_n=10, exclude=()):
        votes = Counter()
        for uid in self.neighbors(profile, int(config["k"])):
            votes.update(self.positives.get(uid, []))
        frame = pd.DataFrame(list(votes.items()), columns=["product_id", "score"])
        return rank_scores(frame, "score", top_n, exclude)

## 4. Preparación del modelo y configuraciones

In [3]:
model = ProfileKNN(ctx)
print("Usuarios de entrenamiento:", model.n_train_users)
print("Perfiles distintos:", len(model.signatures))
CONFIGURACIONES = [{"k": k} for k in [50, 100, 250, 500] if k <= model.n_train_users]
if not CONFIGURACIONES:
    raise ValueError("Para una prueba pequeña, agregá K=5 o K=10 a la lista")

Usuarios de entrenamiento: 502216
Perfiles distintos: 1842


## 5. Ajuste en validación y evaluación final

**Hit Rate@10:** usuarios cuyo producto oculto aparece en el Top-10 / usuarios evaluados.

**NDCG@10:** vale 0 si no aparece; si aparece en posición `r`, vale `1/log2(r+1)`. Con un solo positivo de prueba por usuario, premia que el acierto esté más arriba.

**Cobertura de usuarios:** porcentaje que recibe al menos una recomendación. **Top-10 completo:** porcentaje que recibe diez. **Cobertura de catálogo:** productos distintos recomendados / productos de skincare del catálogo.

Los usuarios sin recomendaciones cuentan como misses. No se eliminan del denominador. El parámetro se elige por HR@10 de validación y, en empate, por NDCG@10. Un empate total conserva la primera configuración de la lista. Test se utiliza después de esa elección.

Se compara con popularidad positiva global calculada exclusivamente sobre train. El promedio de tiempo por consulta es orientativo: puede incluir efectos de caché y no constituye un benchmark de producción.

In [4]:
mejor_config, validacion, metricas_test, detalle_test, baseline_test = sc.run_experiment(
    model, ctx, CONFIGURACIONES)
print("Resultados de validación:")
display(validacion)
print("Configuración elegida:", mejor_config)
print("Resultados finales en test:")
display(pd.DataFrame([metricas_test, baseline_test], index=["Modelo", "Baseline global"]))

Validación: {'k': 50}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 100}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 250}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'k': 500}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Configuración seleccionada: {'k': 250} 
Evaluación test
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 40

,config,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
0,{'k': 50},500,27,0.054,0.029155,1.0,1.0,0.181405,10.0,0.009271
1,{'k': 100},500,25,0.050,0.031770,1.0,1.0,0.140083,10.0,0.007146
2,{'k': 250},500,34,0.068,0.044149,1.0,1.0,0.099174,10.0,0.008516
3,{'k': 500},500,32,0.064,0.038523,1.0,1.0,0.067769,10.0,0.014422


Configuración elegida: {'k': 250}
Resultados finales en test:


,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
Modelo,500,29,0.058,0.027493,1.0,1.0,0.105785,10.0,0.010724
Baseline global,500,37,0.074,0.033161,1.0,1.0,0.004132,10.0,0.006527


## 6. Ejemplo de Top-10 para una persona nueva

Todos los notebooks usan el mismo perfil de ejemplo. Acá no se oculta un producto ni se mide un acierto: se muestra qué devolvería la consulta real. Si no hay evidencia suficiente, la tabla puede tener menos de diez filas.

El ejemplo no reemplaza a la evaluación agregada. No significa que estos productos sean clínicamente adecuados para esa persona.

In [5]:
PERFIL_EJEMPLO = {"skin_type": "oily", "skin_tone": "dark", "eye_color": "blue", "hair_color": "brown"}
perfil, top10 = sc.example_top10(model, ctx, mejor_config, PERFIL_EJEMPLO)
display(pd.DataFrame([perfil]))
display(top10)
print("Productos devueltos:", len(top10))

,skin_type,skin_tone,eye_color,hair_color
0,oily,dark,blue,brown


,position,product_id,product_name,brand_name,secondary_category,price_usd
0,1,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Lip Balms & Treatments,24.0
1,2,P411540,Cicapair Tiger Grass Color Correcting Treatmen...,Dr. Jart+,Moisturizers,52.0
2,3,P422536,Original Skin Matte Moisturizer with Willowherb,Origins,Moisturizers,40.0
3,4,P423688,Daily Microfoliant Exfoliator,Dermalogica,Cleansers,65.0
4,5,P443352,Mini Daily Microfoliant Exfoliator,Dermalogica,Mini Size,18.0
5,6,P471546,Mini Cicapair Tiger Grass Color Correcting Tre...,Dr. Jart+,Mini Size,21.0
6,7,P418218,The Water Cream Oil-Free Pore Minimizing Moist...,Tatcha,Moisturizers,70.0
7,8,P427417,Niacinamide 10% + Zinc 1% Oil Control Serum,The Ordinary,Treatments,6.0
8,9,P454794,Mini The Water Cream,Tatcha,Mini Size,22.0
9,10,P126301,Take The Day Off Cleansing Balm Makeup Remover,CLINIQUE,Cleansers,38.0


Productos devueltos: 10


## 7. Cierre y exportación de evidencia

La conclusión se calcula con tus resultados nuevos. No reutilizar el 6,2 %, 5,6 %, 2,6 % o 1,4 % de los notebooks anteriores: cambiaron el split y las exclusiones.

Analizar: ¿supera al baseline? ¿Cuánto catálogo cubre? ¿Entrega diez productos? ¿Qué información del perfil usa? ¿Qué sesgos mantiene? Un resultado bajo es evidencia útil, no un motivo para ocultar el experimento.

In [6]:
sc.conclusion(metricas_test, baseline_test)
extra = {"algorithm": "ProfileKNN"}

archivo = sc.save_result(ctx, "01_vecinos_por_perfil", mejor_config, validacion, metricas_test,
                         detalle_test, baseline_test, top10, extra)
print("Resultado guardado:", archivo)

HR@10 test: 5.80%; baseline: 7.40%; diferencia: -1.60 puntos porcentuales.
Cobertura de usuarios: 100.00%; catálogo: 10.58%.
En esta muestra no supera al baseline en HR. No declararlo ganador por personalización solamente.
Un hit recupera una preferencia observada; no prueba eficacia ni compatibilidad dermatológica.
Resultado guardado: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63\01_vecinos_por_perfil.json


## 8. Preguntas para entenderlo y defenderlo

1. ¿Qué información usa el algoritmo para ordenar productos?
2. ¿Qué parámetro ajustamos y por qué?
3. ¿Cómo se asegura que el usuario de test no aporte evidencia al ranking?
4. ¿Qué significa un hit y qué no demuestra?
5. ¿Qué aprendimos al compararlo con popularidad global?

Guardá con **Ctrl + S** para conservar todas las salidas y abrí luego `05_comparacion_final.ipynb`.